<a href="https://colab.research.google.com/github/EkaterinaLavlinskaya/My-work/blob/main/modelLogisticRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

МОДЕЛЬ ОБУЧЕНИЯ

In [464]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             mean_squared_error, r2_score)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib

In [465]:
!gdown 1h99toeF7lZ2I3iJwehgKO-QQmDaOe_O3
!gdown 1XL0VTygpZj-ZAuTNRBgApZTPQyNDnT-v

Downloading...
From: https://drive.google.com/uc?id=1h99toeF7lZ2I3iJwehgKO-QQmDaOe_O3
To: /content/game_of_thrones_test.csv
100% 37.3k/37.3k [00:00<00:00, 52.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1XL0VTygpZj-ZAuTNRBgApZTPQyNDnT-v
To: /content/game_of_thrones_train.csv
100% 138k/138k [00:00<00:00, 35.9MB/s]


In [466]:
data = pd.read_csv("/content/game_of_thrones_train.csv", index_col='S.No')
data.head()

,name,title,male,culture,dateOfBirth,mother,father,heir,house,spouse,...,isAliveMother,isAliveFather,isAliveHeir,isAliveSpouse,isMarried,isNoble,age,numDeadRelations,popularity,isAlive
S.No,,,,,,,,,,,,,,,,,,,,,
1,Viserys II Targaryen,NaN,1,NaN,NaN,Rhaenyra Targaryen,Daemon Targaryen,Aegon IV Targaryen,NaN,NaN,...,1.0,0.0,0.0,NaN,0,0,NaN,11,0.605351,0
2,Walder Frey,Lord of the Crossing,1,Rivermen,208.0,NaN,NaN,NaN,House Frey,Perra Royce,...,NaN,NaN,NaN,1.0,1,1,97.0,1,0.896321,1
3,Addison Hill,Ser,1,NaN,NaN,NaN,NaN,NaN,House Swyft,NaN,...,NaN,NaN,NaN,NaN,0,1,NaN,0,0.267559,1
4,Aemma Arryn,Queen,0,NaN,82.0,NaN,NaN,NaN,House Arryn,Viserys I Targaryen,...,NaN,NaN,NaN,0.0,1,1,23.0,0,0.183946,0
5,Sylva Santagar,Greenstone,0,Dornish,276.0,NaN,NaN,NaN,House Santagar,Eldon Estermont,...,NaN,NaN,NaN,1.0,1,1,29.0,0,0.043478,1


In [467]:
data.isnull().sum()

,0
name,0
title,840
male,0
culture,1069
dateOfBirth,1278
mother,1539
father,1535
heir,1536
house,381
spouse,1357


In [468]:
data.describe(include = 'object').T

,count,unique,top,freq
name,1557,1557,Melara Hetherspoon,1
title,717,195,Ser,306
culture,488,51,Northmen,94
mother,18,16,Rhaenyra Targaryen,2
father,22,19,Daemon Targaryen,2
heir,21,20,Jaehaerys Targaryen,2
house,1176,315,House Frey,89
spouse,200,186,Walder Frey,6


In [469]:
data.describe(include=['object', 'int']).T[['count', 'min', 'max']]

,count,min,max
name,1557,NaN,NaN
title,717,NaN,NaN
male,1557.0,0.0,1.0
culture,488,NaN,NaN
mother,18,NaN,NaN
father,22,NaN,NaN
heir,21,NaN,NaN
house,1176,NaN,NaN
spouse,200,NaN,NaN
book1,1557.0,0.0,1.0


In [470]:
missing_summary = data.isnull().sum()

In [471]:
possible_relation_cols = [col for col in data.columns if 'relation' in col.lower() or 'dead' in col.lower() or 'relative' in col.lower()]
print(f"Возможные колонки о родственниках: {possible_relation_cols}")


Возможные колонки о родственниках: ['numDeadRelations']


In [472]:
if 'numDeadRelations' in data.columns:
    data['boolDeadRelations'] = (data['numDeadRelations'] > 0).astype(int)

    data['numDeadRelations'] = data['numDeadRelations'].fillna(0)

In [473]:
if 'age' in data.columns:
    data['age_nan_flag'] = data['age'].isna().astype(int)
    data['age_filled'] = data['age'].fillna(-999)
    data['age'] = data['age'].fillna(-999)

In [474]:
if 'culture' in data.columns:
    cultures_grouped = {
        'Old Nations': ['valyrian', 'first men', 'andal', 'andals', 'rhoynar'],
        'the North': ['northmen', 'northern mountain clans', 'crannogmen'],
        'the Iron Islands': ['ironborn', 'ironborn', 'ironmen'],
        'the Mountain and the Vale': ['valemen', 'vale', 'vale mountain clans', 'sistermen'],
        'the Isles and Rivers': ['riverlands', 'rivermen'],
        'the Rock': ['westerman', 'westermen', 'westerlands'],
        'the Stormlands': ['stormlander', 'stormlands'],
        'the Reach': ['reach', 'reachmen', 'the reach'],
        'Dorne': ['dornish', 'dornishmen', 'dorne'],
        'Essos Nations': ['astapor', 'astapori', 'braavosi', 'braavos', 'tyroshi', 'lysene', 'lyseni',
                          'myrish', 'pentoshi', 'qartheen', 'qarth', 'dothraki',
                          'lhazarene', 'lhazareen','meereen', 'meereenese',
                          'norvoshi', 'qohor', 'summer isles', 'summer islands',
                          'summer islander', 'asshai', "asshai'i", 'norvos', 'ghiscari',
                          'ghiscaricari'],
        'Other Nations': ['ibbenese', 'westeros', 'free folk', 'wildling', 'wildlings', 'naathi']
    }

    cultures_grouped_inverted = {}
    for group, cultures in cultures_grouped.items():
        for culture in cultures:
            cultures_grouped_inverted[culture.lower()] = group

    data['culture_grouped'] = data['culture'].str.lower().map(cultures_grouped_inverted)
    data['culture_grouped_nan'] = data['culture_grouped'].isna().astype(int)
    data['culture_grouped'] = data['culture_grouped'].fillna('Unknown')
    data['culture'] = data['culture'].fillna('Unknown')

In [475]:
cat_cols_with_nan = ['title', 'mother', 'father', 'heir', 'house', 'spouse', 'name']
for col in cat_cols_with_nan:
    if col in data.columns:
        data[f'{col}_nan'] = data[col].isna().astype(int)
        data[col] = data[col].fillna('Unknown')

In [476]:
all_numeric_cols = data.select_dtypes(include=['int64', 'float64']).columns.tolist()
if 'isAlive' in all_numeric_cols:
    all_numeric_cols.remove('isAlive')

for col in all_numeric_cols:
    if data[col].isna().any():
        print(f"Заполняем NaN в {col}: {data[col].isna().sum()} пропусков")

        if f'{col}_nan' not in data.columns:
            data[f'{col}_nan'] = data[col].isna().astype(int)

        data[col] = data[col].fillna(-999)

Заполняем NaN в dateOfBirth: 1278 пропусков
Заполняем NaN в isAliveMother: 1539 пропусков
Заполняем NaN в isAliveFather: 1535 пропусков
Заполняем NaN в isAliveHeir: 1536 пропусков
Заполняем NaN в isAliveSpouse: 1357 пропусков


In [477]:
nan_after = data.isnull().sum()
nan_columns_after = nan_after[nan_after > 0]
if len(nan_columns_after) > 0:
    print("ЕСТЬ NaN:")
    print(nan_columns_after)

In [478]:
 for col in nan_columns_after.index:
        if data[col].dtype in ['int64', 'float64']:
            data[col] = data[col].fillna(-999)
            print(f"  Заполнили {col} значением -999")
        else:
            data[col] = data[col].fillna('Unknown')
            print(f"  Заполнили {col} значением 'Unknown'")
else:
    print("Все NaN обработаны!")

Все NaN обработаны!


In [479]:
categorical_cols = data.select_dtypes(include=['object']).columns.tolist()
if 'isAlive' in categorical_cols:
    categorical_cols.remove('isAlive')

numeric_cols = data.select_dtypes(include=['int64', 'float64']).columns.tolist()
if 'isAlive' in numeric_cols:
    numeric_cols.remove('isAlive')

In [480]:
X = data.drop('isAlive', axis=1)
y = data['isAlive']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [481]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder.fit(X_train[categorical_cols])

X_train_cat = encoder.transform(X_train[categorical_cols])
X_train_cat_df = pd.DataFrame(
    X_train_cat,
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X_train.index
)

In [482]:
X_train_final = pd.concat([
    X_train[numeric_cols].reset_index(drop=True),
    X_train_cat_df.reset_index(drop=True)
], axis=1)

In [483]:

print(f"NaN в X_train_final: {X_train_final.isna().sum().sum()}")



NaN в X_train_final: 0


In [484]:

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_final, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000, random_state=42)

In [485]:
train_pred = model.predict(X_train_final)
print(f"Точность на train: {accuracy_score(y_train, train_pred):.3f}")


Точность на train: 0.856
